# Projeto AVD — Como a taxa Selic impacta sua decisão de investimento?

**Tema:** Finanças — análise comparativa de estratégias de alocação de capital: **Poupança × Renda Fixa × Renda Variável (IBOVESPA)**.

Este notebook documenta e executa o pipeline completo do projeto, fase a fase. O produto final é o dashboard interativo em Streamlit (`main.py`); aqui o objetivo é demonstrar cada etapa de forma reprodutível.

## Fontes de dados

| Fonte | Tipo | O que fornece |
|---|---|---|
| Yahoo Finance (`yfinance`) | API | Histórico do IBOVESPA (`^BVSP`), últimos 2 anos |
| Banco Central do Brasil (SGS, série 11) | API | Meta da taxa Selic histórica |
| InfoMoney e CNN Brasil | **Scraping HTML** (BeautifulSoup, tags `h1`–`h4`) | Manchetes de economia para análise de sentimento |

O scraping é feito de verdade nos portais; IBOVESPA e Selic vêm de API, conforme a orientação do projeto para o tema Finanças. Quando uma fonte está indisponível, o pipeline usa *fallback* sinalizado explicitamente (flag `fonte` no DataFrame).

## Estrutura do pipeline

1. **Fase 1 — Extração** (`src/data_extraction.py`): APIs + scraping
2. **Fase 2 — ETL** (`src/etl.py`): limpeza, agregação mensal, merge, features de ML, sentimento das notícias
3. **Fase 3 — Análise estatística** (`src/analysis.py`): Pearson, ANOVA, descritivas, quartis, outliers, tendência
4. **Fase 4 — ML** (`src/ml_model.py`): Ridge Regression + simulador com regras de negócio
5. **Fase 5 — Dashboard** (`main.py`): Streamlit com princípios de Gestalt

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

# Garante que o notebook encontre o pacote src/ na raiz do projeto
RAIZ = os.path.abspath("")
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)

from src import data_extraction, etl, analysis, ml_model

print("Módulos do projeto importados com sucesso.")

## Fase 1 — Extração de dados

- `fetch_ibovespa("2y")`: fechamentos ajustados do `^BVSP` via Yahoo Finance.
- `fetch_selic(26)`: série 11 do SGS/BCB, agregada para frequência mensal e anualizada. Se a API não responder, usa série sintética e marca `attrs["fonte"] = "sintetica"`.
- `fetch_economic_news(12)`: scraping das páginas de mercados/economia do InfoMoney e da CNN Brasil com BeautifulSoup, percorrendo as tags de título `h1`–`h4`. Se o scraping falhar, usa manchetes de exemplo e marca `attrs["fonte"] = "fallback"`.

In [ ]:
ibov  = data_extraction.fetch_ibovespa("2y")
selic = data_extraction.fetch_selic(26)
news  = data_extraction.fetch_economic_news(12)

print(f"IBOVESPA: {len(ibov)} pregões | de {ibov.index.min().date()} a {ibov.index.max().date()}")
print(f"Selic:    {len(selic)} meses  | fonte: {selic.attrs.get('fonte', 'api')}")
print(f"Notícias: {len(news)} manchetes | fonte: {news.attrs.get('fonte', 'scraping')}")

selic.tail()

## Fase 2 — ETL

`etl.run()` recebe os três DataFrames brutos e produz:

- `ibov_metrics`: retornos diários, volatilidade (janela 21d anualizada), médias móveis MA20/MA50;
- `merged`: base mensal unificando retorno do IBOVESPA e Selic — é a base de toda a análise;
- `ml_features`: features defasadas (`selic_lag1`, `selic_delta`, `ibov_lag1`) para o modelo;
- `news` / `news_summary`: manchetes classificadas por sentimento e tema dominante.

Tudo opera em memória, sem persistência em disco — os dados são sempre buscados das fontes.

In [ ]:
resultado = etl.run(ibov, selic, news)

merged       = resultado["merged"]
ibov_metrics = resultado["ibov_metrics"]
ml_features  = resultado["ml_features"]
news_summary = resultado["news_summary"]

print(f"Base mensal unificada: {len(merged)} observações")
print(f"Resumo das notícias:   {news_summary}")

merged.tail()

## Fase 3 — Análise estatística

Hipótese econômica: **Selic alta → renda fixa atrativa → pressão vendedora na bolsa** (correlação negativa esperada entre Selic e retorno mensal do IBOVESPA).

> **Nota sobre significância:** o período analisado tem ~24 observações mensais. Com amostra desse tamanho, resultados **não significativos (p ≥ 0,05)** em Pearson e ANOVA são estatisticamente esperados pela baixa potência do teste — é uma limitação do horizonte de dados, não do código.

In [ ]:
# Correlação de Pearson — Selic x retorno mensal do IBOVESPA
corr = analysis.pearson_selic_ibov(merged)
print(f"Pearson r = {corr['r']} | R² = {corr['r2']} | p = {corr['p_value']} | n = {corr['n']}")
print(f"Significativo (p < 0,05)? {corr['significativo']}")
print(f"Interpretação: {corr['interpretacao']}\n")

# ANOVA — retorno mensal por regime de Selic
anova = analysis.anova_retorno_por_regime_selic(merged)
print(f"ANOVA F = {anova['f_stat']:.4f} | p = {anova['p_value']} | grupos = {anova['grupos_validos']}")
print(f"Significativo? {anova['significativo']}")
print(f"Interpretação: {anova['interpretacao']}")

In [ ]:
# Estatísticas descritivas dos retornos diários (%)
desc = analysis.stats_descritivas(ibov_metrics["retorno_diario"] * 100)
print("Descritivas (retorno diário %):", desc, "\n")

# Quartis do retorno mensal
print("Quartis do retorno mensal (%):")
print(analysis.analise_quartis(merged["ibov_retorno_mensal"]), "\n")

# Outliers (IQR + Z-Score)
outliers = analysis.resumo_outliers(merged)
print(f"Outliers detectados: {len(outliers)}")

# Tendência do IBOVESPA (regressão linear sobre o tempo)
tend = analysis.tendencia_ibovespa(ibov_metrics)
print(f"Tendência: {tend['tendencia']} | inclinação = {tend['inclinacao']:.2f} pts/dia | R² = {tend['r2']}")

## Fase 4 — Machine Learning

**Ridge Regression** prevendo o retorno mensal do IBOVESPA a partir da Selic:

- **Features:** `selic_anual`, `selic_lag1`, `selic_delta`, `ibov_lag1`
- **Target:** `ibov_retorno_mensal` (%)
- **Split temporal 80/20** — o teste usa apenas os meses mais recentes, sem *data leakage*.

> **Limitação amostral (esperada):** a base tem ~22 observações mensais e ~5 no teste. Um **R² negativo** indica que o modelo não generaliza nesse horizonte curto — limitação de dados, não de implementação. Por isso, a recomendação final do simulador combina a predição do ML com **regras de negócio** derivadas da análise histórica (1999–2024).

In [ ]:
res = ml_model.treinar_modelo(ml_features)

print(f"R² no teste:  {res['r2']}")
print(f"MAE no teste: {res['mae']} pp de retorno mensal")
print(f"Observações:  {res['n_treino']} treino / {res['n_teste']} teste")
print(f"Coeficientes: {res['coeficientes']}")

if (isinstance(res["r2"], float) and not np.isnan(res["r2"]) and res["r2"] < 0) or res["n_teste"] < 8:
    print("\nAVISO: amostra pequena — R² negativo/baixo é esperado neste horizonte (~24 meses).")
    print("A recomendação do simulador combina o ML com regras de negócio.")

In [ ]:
# Simulação: R$ 10.000 por 24 meses, perfil de risco médio, Selic atual da base
selic_atual = float(merged["selic_anual"].iloc[-1])
ibov_lag1   = float(ml_features["ibov_lag1"].dropna().iloc[-1])
pred_mensal = ml_model.prever_retorno_ibov(res, selic_atual, ibov_lag1)

sim = ml_model.simular_investimento(
    selic_anual=selic_atual,
    valor=10_000.0,
    periodo_meses=24,
    tolerancia="media",
    ibov_pred_mensal=pred_mensal,
)

print(f"Selic usada: {selic_atual*100:.2f}% a.a. | Retorno IBOV previsto: {pred_mensal:.4f}% a.m.\n")
for estrategia, montante in sim["montantes"].items():
    print(f"  {estrategia:<15} R$ {montante:>12,.2f}  ({sim['taxas_mensais'][estrategia]:.4f}% a.m.)")

print(f"\nRecomendação: {sim['recomendacao']} (confiança: {sim['confianca']})")
print(f"Justificativa: {sim['justificativa']}")

## Fase 5 — Dashboard e princípios de Gestalt

O produto final é o dashboard Streamlit, executado com:

```bash
streamlit run main.py
```

Abas: **Contexto** (storytelling Selic × IBOVESPA, notícias raspadas), **Simulador**, **Análise Estatística** e **ML Recomendação**.

### Princípios de Gestalt aplicados ao design

- **Proximidade:** KPIs relacionados agrupados em blocos no topo do dashboard.
- **Similaridade:** cores fixas por estratégia — Poupança em verde, Renda Fixa em azul, Renda Variável em roxo.
- **Continuidade:** séries temporais traçadas como linhas contínuas, sem marcadores isolados.
- **Figura/fundo:** tema escuro (`#0e1117`) destaca os elementos de dados sobre o plano de fundo.
- **Pregnância:** layout enxuto, com poucas seções por aba e hierarquia visual clara.

## Conclusões e limitações

**Conclusões**

- O pipeline integra três tipos de fonte (API financeira, API governamental e scraping HTML) em uma base mensal unificada, sem persistência em disco.
- A relação Selic × IBOVESPA observada segue a direção esperada pela teoria econômica, ainda que sem significância estatística no horizonte disponível.
- O simulador entrega recomendações coerentes por perfil de risco, combinando a predição do modelo com regras de negócio.

**Limitações (transparência acadêmica)**

1. **Amostra curta:** ~24 meses de dados mensais limitam a potência dos testes estatísticos e a generalização do modelo (R² de teste pode ser negativo).
2. **Fallbacks:** quando API do BCB ou scraping falham, o pipeline usa dados sintéticos/de exemplo — sempre sinalizados na interface e na flag `fonte`.
3. **Modelo simples por escolha:** Ridge Regression com poucas features é interpretável e adequado ao escopo da disciplina; não é um modelo de produção para decisão financeira real.